# 04b. Línea Base TF-IDF

**Objetivo:** entrenar baseline lexical-estadístico sobre texto clínico.
**Entradas (inputs):** `data/splits/train_denoised.csv` y `data/splits/<split>_denoised.csv`.
**Salidas (outputs):** `data/tfidf_eval.csv`, `data/tfidf_classification_report.csv`, `data/tfidf_predicciones_<split>.csv`.
**Notebook anterior:** `notebooks/pipeline/03_denoising_reglas_core.ipynb`.
**Notebook siguiente:** `notebooks/analysis/05_brecha_lexica_co_core_py.ipynb` y `notebooks/pipeline/08_resultados_hibrido_vs_lineas_base.ipynb`.

> **Aclaración metodológica:** esta línea base trabaja sobre texto y no resuelve por sí sola variación regional/jopará; esa brecha se justifica en el notebook 05.
> **Contrato de comparación:** la evaluación usa el mismo subconjunto denoised que el híbrido (`BASELINE_EVAL_ON=dev` por defecto).


## Técnicas, herramientas y librerías de esta etapa

- **Técnica principal:** baseline textual lexical-estadístico con `TF-IDF` y `LinearSVC`.
- **Herramientas/librerías:** `scikit-learn` (`TfidfVectorizer`, `LinearSVC`, métricas), `pandas`, `numpy`.
- **Por qué es adecuada aquí:** sigue siendo un baseline fuerte, barato y muy competitivo para un corpus clínico relativamente pequeño y desbalanceado. Además permite mostrar qué tanto se puede lograr sin contexto profundo ni reglas clínicas.
- **Limitación:** trata el texto como patrón léxico, no resuelve bien variación regional, abreviaturas nuevas ni contexto clínico largo.
- **Alternativa mejor cuando se necesita contexto:** baselines Transformer como `ROBERTA_CLINICAL`; aun así, `TF-IDF` debe mantenerse porque funciona como control metodológico simple y fuerte.


In [ ]:
# ===============================================================
# Importaciones y configuración de rutas
# ===============================================================
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import re
import unicodedata
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline

# Importar utilidades compartidas
try:
    from utils_shared import setup_paths, load_splits, calculate_metrics, get_cv_splitter
    paths = setup_paths()
    DATA_PATH = paths['DATA_PATH']
    SPLITS_PATH = paths['SPLITS_PATH']
    print("Usando utils_shared.py")
except ImportError:
    print("Error: No se encontró utils_shared.py. Verifica que estás en el directorio correcto.")
    raise

EVAL_ON = os.getenv('BASELINE_EVAL_ON', 'dev').strip().lower()
if EVAL_ON not in {'dev', 'test'}:
    raise ValueError(f"BASELINE_EVAL_ON inválido: {EVAL_ON}. Use 'dev' o 'test'.")
print('EVAL_ON:', EVAL_ON)


## 1) Carga de datos y preprocesamiento
Se entrena con `train_denoised.csv` y se evalúa con `<split>_denoised.csv` para mantener el mismo universo de evaluación del híbrido.


In [ ]:
# Cargar conjuntos de datos procesados
try:
    train_path = SPLITS_PATH / 'train_denoised.csv'
    eval_path = SPLITS_PATH / f'{EVAL_ON}_denoised.csv'

    df_train = pd.read_csv(train_path)
    df_eval = pd.read_csv(eval_path)

    print(f"Entrenamiento (denoised): {len(df_train)} casos")
    print(f"Evaluación ({EVAL_ON}_denoised): {len(df_eval)} casos")
except FileNotFoundError:
    print("Error: No se encontraron los datasets denoised. Ejecuta pipeline/03_denoising_reglas_core.ipynb primero.")
    raise

# Preprocesamiento textual para TF-IDF
RE_MULTI = re.compile(r'(.){2,}')

def clean_text_ml(s: str) -> str:
    if pd.isna(s):
        return ""
    s = str(s).lower().strip()
    s = unicodedata.normalize("NFC", s)
    s = RE_MULTI.sub(r'', s)
    s = re.sub(r"[^a-z0-9áéíóúüñ\s.,!?:/\-]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    # Marca negaciones simples: "no tengo" -> "no_tengo"
    s = re.sub(r"no\s+([a-záéíóúüñ]{2,})", r"no_", s)
    return s

print("Preprocesando textos...")
df_train['texto_ml'] = df_train['texto'].map(clean_text_ml)
df_eval['texto_ml'] = df_eval['texto'].map(clean_text_ml)

X_train = df_train['texto_ml']
y_train = df_train['etiqueta']
X_eval = df_eval['texto_ml']
y_eval = df_eval['etiqueta']


## 2) Entrenamiento y evaluación


In [ ]:
# Flujo base: TF-IDF de caracteres + LinearSVC
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        analyzer='char_wb',
        ngram_range=(3, 5),
        min_df=2,
        max_features=10000
    )),
    ('clf', LinearSVC(class_weight='balanced', random_state=42))
])

print("Entrenando modelo...")
pipeline.fit(X_train, y_train)

print(f"Evaluando en {EVAL_ON}_denoised...")
y_pred = pipeline.predict(X_eval)

# Calcular métricas
metrics = calculate_metrics(y_eval, y_pred)

print("=" * 60)
print("Resultados TF-IDF + LinearSVC")
print("=" * 60)
print(f"Macro F1: {metrics['f1_macro']:.4f}")
print(f"Macro Precision: {metrics['precision_macro']:.4f}")
print(f"Macro Recall: {metrics['recall_macro']:.4f}")
print()
print(metrics['report'])

# Exportar resultados
eval_df = pd.DataFrame([{
    'modelo': 'tfidf',
    'f1_macro': metrics['f1_macro'],
    'precision_macro': metrics['precision_macro'],
    'recall_macro': metrics['recall_macro'],
    'accuracy': metrics['accuracy'],
    'n_train': len(X_train),
    'n_eval': len(X_eval),
    'n_dev': len(X_eval),  # compatibilidad hacia atrás
    'eval_split': EVAL_ON,
}])
eval_df.to_csv(DATA_PATH / 'tfidf_eval.csv', index=False)
print(f"Archivo exportado: {DATA_PATH / 'tfidf_eval.csv'}")

# Exportar reporte completo
report_df = pd.DataFrame(metrics['report_dict']).transpose()
report_df.to_csv(DATA_PATH / 'tfidf_classification_report.csv')
print(f"Archivo exportado: {DATA_PATH / 'tfidf_classification_report.csv'}")

# Exportar predicciones para trazabilidad por row_id
pred_df = pd.DataFrame({
    'row_id': df_eval['row_id'].astype(int),
    'y_true': y_eval.astype(str),
    'y_pred': pd.Series(y_pred).astype(str),
})
pred_path = DATA_PATH / f'tfidf_predicciones_{EVAL_ON}.csv'
pred_df.to_csv(pred_path, index=False)
print(f"Archivo exportado: {pred_path}")
